In [44]:
from astroquery.sdss import SDSS
from astroquery.ipac.irsa import Irsa
from astroquery.vizier import Vizier
from astropy import coordinates as coord
from astropy import units as u
from astroquery.ipac.ned import Ned
import numpy as np
import pandas as pd

In [45]:
# Coordenadas de ejemplo
ra, dec = 1.7376141,0.8571812 # en grados
pos = coord.SkyCoord(ra, dec, unit=(u.deg, u.deg), frame='icrs')

# ---- 1. SDSS (u,g,r,i,z) ----
sdss_data = SDSS.query_region(
    pos,
    radius=0.15 * u.arcsec,   # <--- este es el argumento que faltaba
    spectro=False,
    photoobj_fields=[
        'ra','dec',
        'modelMag_u','modelMagErr_u',
        'modelMag_g','modelMagErr_g',
        'modelMag_r','modelMagErr_r',
        'modelMag_i','modelMagErr_i',
        'modelMag_z','modelMagErr_z'
    ]
)

sdss_data

objID,ra,dec,modelMag_u,modelMagErr_u,modelMag_g,modelMagErr_g,modelMag_r,modelMagErr_r,modelMag_i,modelMagErr_i,modelMag_z,modelMagErr_z
uint64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
1237645879019765919,1.73762334596665,0.85717687320632,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0
1237645879019831364,1.73757747583687,0.857196920707493,19.13904,0.02542134,18.5538,0.008198288,18.76767,0.01176654,18.21628,0.01195736,18.57623,0.05468069


In [3]:
# Coordenadas de ejemplo
ra, dec = 1.7376141,0.8571812 # en grados
pos = coord.SkyCoord(ra, dec, unit=(u.deg, u.deg), frame='icrs')

# ---- 4. WISE (W1–W4) ----
wise = Irsa.query_region(pos, 
                         catalog='allwise_p3as_psd', 
                         spatial='Cone', 
                         radius=0.15*u.arcsec)
wise['w1mpro','w1sigmpro','w2mpro','w2sigmpro','w3mpro','w3sigmpro','w4mpro','w4sigmpro']


w1mpro,w1sigmpro,w2mpro,w2sigmpro,w3mpro,w3sigmpro,w4mpro,w4sigmpro
mag,mag,mag,mag,mag,mag,mag,mag
float32,float32,float32,float32,float32,float32,float32,float32
16.286,0.074,15.521,0.121,10.896,0.132,7.271,0.123


In [46]:
result_table = Ned.get_table("UM 199", table='photometry')


result_table[result_table['Observed Passband']=='FUV (GALEX) AB']

No.,Observed Passband,Photometry Measurement,Uncertainty,Units,Frequency,Flux Density,Upper limit of uncertainty,Lower limit of uncertainty,Upper limit of Flux Density,Lower limit of Flux Density,NED Uncertainty,NED Units,Refcode,Significance,Published frequency,Frequency Mode,Coordinates Targeted,Spatial Mode,Qualifiers,Comments
,,,,,Hz,Jy,,,,,,,,,,,,,,
int32,object,float64,object,object,float64,float64,float64,float64,float64,float64,object,object,object,object,object,object,object,object,object,object
1,FUV (GALEX) AB,19.6186,+/-0.0806745,mag,1950000000000000.0,5.16e-05,3.83e-06,3.83e-06,--,--,+/-3.83E-06,Jy,2012GASC..C...0000S,uncertainty,1538.6 A,Broad-band measurement,1.7375447045308 0.8572865393478 (J2000),Flux integrated from map,Kron flux in elliptical aperture,From new raw data
2,FUV (GALEX) AB,20.1013,+/-0.113131,mag,1950000000000000.0,3.31e-05,3.45e-06,3.45e-06,--,--,+/-3.45E-06,Jy,2012GASC..C...0000S,uncertainty,1538.6 A,Broad-band measurement,1.7375447045308 0.8572865393478 (J2000),Flux in fixed aperture,Flux in 7.5 arcsec diameter aperture,From new raw data
3,FUV (GALEX) AB,19.5745,+/-0.0293216,mag,1950000000000000.0,5.37e-05,1.45e-06,1.45e-06,--,--,+/-1.45E-06,Jy,2012GMSC..C...0000S,uncertainty,1538.6 A,Broad-band measurement,1.7377864753749 0.8572492597596 (J2000),Flux integrated from map,Kron flux in elliptical aperture,From new raw data
4,FUV (GALEX) AB,19.9717,+/-0.0342690,mag,1950000000000000.0,3.73e-05,1.18e-06,1.18e-06,--,--,+/-1.18E-06,Jy,2012GMSC..C...0000S,uncertainty,1538.6 A,Broad-band measurement,1.7377864753749 0.8572492597596 (J2000),Flux in fixed aperture,Flux in 7.5 arcsec diameter aperture,From new raw data


In [5]:
np.unique(result_table['Observed Passband'])

FUV (GALEX) AB
H{beta} line
NUV (GALEX) AB
W1 (WISE)
W2 (WISE)
W3 (WISE)
W4 (WISE)
g (SDSS CModel) AB
g (SDSS Model) AB
g (SDSS PSF) AB
g (SDSS Petrosian)AB


In [ ]:
import os
HOME = os.path.expanduser("~") + '/gdrive/DataHII/'

def magAB_to_flux(mag, mag_err):
    f = 10**(-0.4 * (mag - 8.90))
    ferr = f * np.log(10)/2.5 * mag_err
    return f, ferr

WISE_ZP = {
    'W1': 309.540,
    'W2': 171.787,
    'W3': 31.674,
    'W4': 8.363
}

def vega_to_jy(mag, mag_err, band):
    """Convierte magnitudes Vega de WISE a flux (Jy) con error."""
    f0 = WISE_ZP[band.upper()]
    f = f0 * 10**(-0.4 * mag)
    ferr = f * np.log(10)/2.5 * mag_err
    return f, ferr



def GALEX_data(table,filter_band):
    
    TABLE = table[(table['Observed Passband']==filter_band)]

    check = np.unique(TABLE['Significance'])

    # OJO con esto despues, hay un valor malo aqui es el "SBS 0926+606A" para el FUV
    if ((np.where(check == 'no uncertainty reported'))!= 0)  and (len(np.unique(TABLE['Uncertainty']))==1):
        #CHOSEN = table[table['Significance']=='no uncertainty reported']
        flx = TABLE['Photometry Measurement'][0]

        fx_mJy, fxerr_mJy = magAB_to_flux(flx, 0.0)

        #return flx, flx_err
        #return fx_mJy, fxerr_mJy
        fxerr_mJy = np.random.uniform(low=0,high=0.018)
        return fx_mJy * 1e3 , fxerr_mJy * 1e3
    
    else:
        TABLE = table[(table['Observed Passband']==filter_band) & (table['Qualifiers']=='Kron flux in elliptical aperture')]
        errors = np.array(TABLE['Uncertainty'])

        Err_min = np.min(errors)

        CHOSEN = TABLE[TABLE['Uncertainty']==Err_min]


        flx = CHOSEN['Photometry Measurement'][0]
        flx_err = float(CHOSEN['Uncertainty'][0].replace("+/-",""))

        fx_mJy, fxerr_mJy = magAB_to_flux(flx, flx_err)

        #return flx, flx_err
        #return fx_mJy, fxerr_mJy
        return fx_mJy * 1e3 , fxerr_mJy * 1e3



def SDSS_data(table,filter_band):
    TABLE = table[(table['Observed Passband']==filter_band)]

    Ref = np.array(TABLE['Refcode'])

    Older_ref = np.max(Ref)

    CHOSEN = TABLE[TABLE['Refcode']==Older_ref]


    flx = CHOSEN['Photometry Measurement'][0]
    flx_err = float(CHOSEN['Uncertainty'][0].replace("+/-",""))

    fx_mJy, fxerr_mJy = magAB_to_flux(flx, flx_err)

    #return flx, flx_err
    #return fx_mJy, fxerr_mJy
    return fx_mJy * 1e3 , fxerr_mJy * 1e3










def WISE_data(table,filter_band):

    band = filter_band[0:2]

    TABLE = table[(table['Observed Passband']==filter_band) & (table['Qualifiers']=='Profile-fit')]

    CHOSEN = TABLE


    flx = CHOSEN['Photometry Measurement'][0]

    str_removal = CHOSEN['Uncertainty'][0].replace("+/-","")
    str_removal = str_removal.replace(">","")

    flx_err = float(str_removal)
    #flx_err = float(CHOSEN['Uncertainty'][0].replace(">",""))

    if ('-' in str(flx)) or ('-' in str(flx_err)):
        return np.nan , np.nan
    else:
        fx_mJy, fxerr_mJy = vega_to_jy(flx, flx_err,band)
        #return flx, flx_err
        #return fx_mJy, fxerr_mJy
        return fx_mJy * 1e3 , fxerr_mJy * 1e3

In [7]:
# Sacando los datos para CIGALE

DF = pd.read_csv(HOME+'/SPS/HIIGs_aladin.csv')

head = "# id redshift galex.FUV galex.FUV_err galex.NUV galex.NUV_err "
head = head + "sloan.sdss.u sloan.sdss.u_err sloan.sdss.g sloan.sdss.g_err sloan.sdss.r sloan.sdss.r_err sloan.sdss.i sloan.sdss.i_err sloan.sdss.z sloan.sdss.z_err "
head = head + "wise.W1 wise.W1_err wise.W2 wise.W2_err wise.W3 wise.W3_err wise.W4 wise.W4_err \n"

with open("cigale_HIIG_sample.txt", 'w') as file:
    file.write(head)


    for n in range(len(DF)):

        name = DF['Name'][n]

        filters = ['FUV (GALEX) AB', 'NUV (GALEX) AB','u (SDSS Model) AB','g (SDSS Model) AB','r (SDSS Model) AB','i (SDSS Model) AB','z (SDSS Model) AB',
                  'W1 (WISE)','W2 (WISE)','W3 (WISE)','W4 (WISE)']
        NED_table = Ned.get_table(name, table='photometry')
        ned_filters = np.unique(NED_table['Observed Passband'])

        idx,z = '',0
        fuv,nuv,u,g,r,i,z,W1,W2,W3,W4 = 0,0,0,0,0,0,0,0,0,0,0
        e1,e2,e3,e4,e5,e6,e7,e8,e9,e10,e11 = 0,0,0,0,0,0,0,0,0,0,0

        cond = (filters[0] in ned_filters) and (filters[1] in ned_filters) and (filters[2] in ned_filters) and (filters[3] in ned_filters) and (filters[4] in ned_filters) \
            and (filters[5] in ned_filters) and (filters[6] in ned_filters) and (filters[7] in ned_filters) and (filters[8] in ned_filters) and (filters[9] in ned_filters)

        
        if cond:
            idx = str(DF['N'][n]) + '_' + str(DF['Name'][n]).replace(' ','')
            z = DF['Z'][n]
            print(idx)

            fuv,e1 = GALEX_data(NED_table,filters[0])
            nuv,e2 = GALEX_data(NED_table,filters[1])

            u,e3 = SDSS_data(NED_table,filters[2])
            g,e4 = SDSS_data(NED_table,filters[3])
            r,e5 = SDSS_data(NED_table,filters[4])
            i,e6 = SDSS_data(NED_table,filters[5])
            z,e7 = SDSS_data(NED_table,filters[6])

            W1,e8 = WISE_data(NED_table,filters[7])
            W2,e9 = WISE_data(NED_table,filters[8])
            W3,e10 = WISE_data(NED_table,filters[9])
            W4,e11 = WISE_data(NED_table,filters[10])



            output = f"{idx} {z} {fuv} {e1} {nuv} {e2} "
            output = output + f"{u} {e3} {g} {e4} {r} {e5} {i} {e6} {z} {e7} "
            output = output + f"{W1} {e8} {W2} {e9} {W3} {e10} {W4} {e11} \n"

            file.write(output)
            




0_UM199
1_UM282
2_UM336
4_SBS0934+546
5_UM570
6_UM162
10_SHOC391
11_WAS69
12_WAS17
14_WISEAJ095227.57+322810.3
15_WISEAJ094252.78+354725.7
16_WISEAJ105040.86+342946.3
17_TOLOLO1057+083
20_SHOC143
21_SHOC220
22_SHOC316
23_SHOC008
24_SBS0813+521
25_WISEAJ094809.88+425713.1
28_SHOC022
29_WISEAJ121329.48+114057.2
31_WISEAJ130119.26+123959.7
32_WISEAJ104653.98+134646.4
33_SHOC293
34_SBS0926+606A
36_SHOC282
38_SHOC014
39_SHOC043
40_SHOC113
41_SHOC592
42_SHOC590
43_SHOC247
44_WISEAJ231442.11+010621.6
46_SHOC084
47_SHOC115
48_SHOC133
51_SHOC153
52_SHOC174
53_SHOC203
57_SHOC486
58_SHOC584
59_SHOC586
60_SHOC595
62_WISEAJ104457.84+035312.9
63_WISEAJ105326.00+043014.5
65_WISEAJ220802.85+131334.5
66_WISEAJ212043.94+010006.9
67_WISEAJ104755.92+073951.2
68_WISEAJ214350.85-072003.9
69_WISEAJ092540.91+063116.8
71_WISEAJ092749.17+084037.1
72_WISEAJ101036.62+641242.6
73_WISEAJ100746.48+025229.1
74_WISEAJ222510.11-001153.1
76_WISEAJ105210.40+032712.7
78_WISEAJ102429.21+052451.0
79_WISEAJ090531.04+033530.3

In [8]:
#Numero de galaxias con datos completos

DF = pd.read_csv(HOME+'/SPS/HIIGs_aladin.csv')

HIIG_names = np.array(DF['Name'])

print('Name | FUV | NUV | u | g | r | i | z | W1 | W2 | W3 | W4 | ALL |')

sep = ' | '

name_id = 0

Found = 0

for name in HIIG_names:
    result_table = Ned.get_table(name, table='photometry')
    filters = np.unique(result_table['Observed Passband'])

    c = 0 

    tab = str(name_id) + '_id' + sep

    #FUV
    if 'FUV (GALEX) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #NUV
    if 'NUV (GALEX) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #u
    if 'u (SDSS Model) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #g
    if 'g (SDSS Model) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #r
    if 'r (SDSS Model) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #i
    if 'i (SDSS Model) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #z
    if 'z (SDSS Model) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #W1
    if 'W1 (WISE)' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #W2
    if 'W2 (WISE)' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #W3
    if 'W3 (WISE)' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #W4
    if 'W4 (WISE)' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    


    if c == 11:
        tab += '  All found' + sep
        Found += 1
        print(tab + name)
    else:
        tab += str(0) + sep


    

    name_id += 1

print(f'\n Found complete:{Found}')    
    


Name | FUV | NUV | u | g | r | i | z | W1 | W2 | W3 | W4 | ALL |
0_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | UM 199
1_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | UM 282
2_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | UM 336
4_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | SBS 0934+546
5_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | UM 570
6_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | UM 162
10_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | SHOC 391
11_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | WAS 69
12_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | WAS 17
14_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | WISEA J095227.57+322810.3
15_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | WISEA J094252.78+354725.7
16_id | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |   All found | WISEA J105040.86+342946.3


In [9]:
from astroquery.vizier import Vizier
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
Vizier.clear_cache()

In [25]:
DF

,N,Name,ra,dec,Z,PLATE,MJD,FIBERID
0,0,UM 199,1.737614,0.857181,0.073682,388,51793,457
1,1,UM 282,12.947112,0.161124,0.037556,394,51913,472
2,2,UM 336,23.436174,0.953078,0.019236,400,51820,441
3,3,MRK 0627,131.643397,36.438978,0.010634,934,52672,369
4,4,SBS 0934+546,144.556301,54.473618,0.102102,556,51991,224
...,...,...,...,...,...,...,...,...
116,116,WISEA J103226.97+271755.3,158.112320,27.298679,0.192490,2353,53794,585
117,117,WISEA J101136.07+263027.5,152.900317,26.507660,0.054666,2347,53757,501
118,118,WISEA J162152.58+151855.9,245.469098,15.315537,0.034341,2208,53880,306
119,119,WISEA J090506.84+223834.7,136.278600,22.642759,0.125548,2284,53708,545


In [42]:
Vizier.ROW_LIMIT = 10000 
for i in range(len(DF)):
    ra, dec = DF['ra'][i], DF['dec'][i]
    pos = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")

    cat_id = "II/319/las9"

    res = Vizier.query_region(pos, radius=1*u.arcsec, catalog=cat_id)

    if res:
        tab = res[0]
        df = tab.to_pandas()
        LEN = len(df)
    else:
        print("Sin detección en UKIDSS dentro del radio dado  ",i)






    


Sin detección en UKIDSS dentro del radio dado   3
Sin detección en UKIDSS dentro del radio dado   4
Sin detección en UKIDSS dentro del radio dado   7
Sin detección en UKIDSS dentro del radio dado   8
Sin detección en UKIDSS dentro del radio dado   9
Sin detección en UKIDSS dentro del radio dado   12
Sin detección en UKIDSS dentro del radio dado   13
Sin detección en UKIDSS dentro del radio dado   14
Sin detección en UKIDSS dentro del radio dado   15
Sin detección en UKIDSS dentro del radio dado   16
Sin detección en UKIDSS dentro del radio dado   19
Sin detección en UKIDSS dentro del radio dado   21
Sin detección en UKIDSS dentro del radio dado   22
Sin detección en UKIDSS dentro del radio dado   23
Sin detección en UKIDSS dentro del radio dado   24
Sin detección en UKIDSS dentro del radio dado   25
Sin detección en UKIDSS dentro del radio dado   26
Sin detección en UKIDSS dentro del radio dado   34
Sin detección en UKIDSS dentro del radio dado   38
Sin detección en UKIDSS dentro del r

In [35]:
# Configuración
Vizier.ROW_LIMIT = 10000  # puedes aumentar si buscas muchos objetos
#Vizier.columns = ["RAJ2000", "DEJ2000", "Ymag", "Jmag", "Hmag", "Kmag", "e_Ymag", "e_Jmag", "e_Hmag", "e_Kmag"]

# Coordenadas del objeto
ra, dec = 12.947112,0.16112404 # grados
pos = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")

# Catálogo UKIDSS LAS en VizieR
cat_id = "II/319/"  # DR9 LAS (hay DR10/DR11 también)

# Consulta por radio
res = Vizier.query_region(pos, radius=1*u.arcsec, catalog=cat_id)

if res:
    tab = res[0]
    df = tab.to_pandas()
    df
else:
    print("Sin detección en UKIDSS dentro del radio dado.")


df

,ULAS,m,RAJ2000,DEJ2000,Ymag,e_Ymag,Jmag1,e_Jmag1,Jmag2,e_Jmag2,Hmag,e_Hmag,Kmag,e_Kmag,Epoch,pmRA,pmDE,cl
0,J005147.32+000939.9,1,12.947188,0.161089,18.503,0.043,18.181,0.064,NaN,NaN,17.636999,0.057,17.327,0.094,2005.7657,NaN,NaN,1


In [11]:
from astroquery.vizier import Vizier
vizier = Vizier() # this instantiates Vizier with its default parameters
catalog_list = vizier.find_catalogs('ukidss')

for k, v in catalog_list.items():
    print(k, ":", v.description)

II/316 : UKIDSS-DR6 Galactic Plane Survey (Lucas+ 2012)
II/319 : UKIDSS-DR9 LAS, GCS and DXS Surveys (Lawrence+ 2012)


In [12]:
from astroquery.vizier import Vizier
vizier = Vizier() # this instantiates Vizier with its default parameters
catalog_list = vizier.find_catalogs('wise')

for k, v in catalog_list.items():
    print(k, ":", v.description)

II/311 : WISE All-Sky Data Release (Cutri+ 2012)
II/328 : AllWISE Data Release (Cutri+ 2013)
II/363 : The band-merged unWISE Catalog (Schlafly+, 2019)
II/365 : The CatWISE2020 catalog (updated version 28-Jan-2021) (Marocco+, 2021)


In [14]:
vizier.ROW_LIMIT = 50
UKIDSS = vizier.get_catalogs("II/319/las9")
Uk_cat = UKIDSS[0]
Uk_cat = Uk_cat.to_pandas()
Uk_cat

,ULAS,m,RAJ2000,DEJ2000,Ymag,e_Ymag,Jmag1,e_Jmag1,Jmag2,e_Jmag2,Hmag,e_Hmag,Kmag,e_Kmag,Epoch,pmRA,pmDE,cl
0,J131601.51-001710.5,1,199.006326,-0.286277,NaN,NaN,18.550,0.095,NaN,NaN,17.636000,0.100,16.819000,0.080,2005.4480,NaN,NaN,1
1,J131606.65-001737.0,1,199.027721,-0.293624,NaN,NaN,NaN,NaN,NaN,NaN,18.854000,0.194,18.184000,0.181,2009.1556,NaN,NaN,-2
2,J131607.02-001721.4,2,199.029261,-0.289282,18.007000,0.024,NaN,NaN,NaN,NaN,16.862000,0.032,16.669001,0.046,2009.4343,NaN,NaN,-1
3,J131607.02-001721.4,1,199.029267,-0.289279,NaN,NaN,17.429,0.035,NaN,NaN,16.959000,0.054,16.806000,0.080,2005.4480,NaN,NaN,-1
4,J131605.36-001725.9,1,199.022372,-0.290555,NaN,NaN,NaN,NaN,NaN,NaN,18.785000,0.182,18.122000,0.171,2009.1556,NaN,NaN,-1
5,J131605.36-001725.8,2,199.022367,-0.290509,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.117001,0.265,2005.3768,NaN,NaN,-1
6,J131610.62-001658.7,1,199.044253,-0.282975,NaN,NaN,18.749,0.114,NaN,NaN,NaN,NaN,17.797001,0.198,2005.4480,NaN,NaN,-2
7,J131610.63-001658.6,2,199.044322,-0.282950,19.334000,0.074,NaN,NaN,NaN,NaN,18.393000,0.127,NaN,NaN,2009.4343,NaN,NaN,1
8,J131611.00-001656.6,2,199.045860,-0.282390,19.186001,0.065,NaN,NaN,NaN,NaN,18.112000,0.098,18.047001,0.159,2009.4343,NaN,NaN,-1
9,J131611.00-001656.5,1,199.045855,-0.282371,NaN,NaN,18.631,0.103,NaN,NaN,18.118999,0.156,17.756001,0.191,2005.4480,NaN,NaN,-1


In [15]:
Vizier(catalog="II/319").get_catalog_metadata()

title,authors,abstract,origin_article,webpage,created,updated,waveband,doi
object,object,object,object,object,object,object,object,object
"UKIDSS-DR9 LAS, GCS and DXS Surveys","Lawrence A.; Warren S.J.; Almaini O.; Edge A.C.; Hambly N.C.; Jameson R.F.,Lucas P.; Casali M.; Adamson A.; Dye S.; Emerson J.P.; Foucaud S.,Hewett P.; Hirst P.; Hodgkin S.T.; Irwin M.J.; Lodieu N.; McMahon R.G.,Simpson C.; Smail I.; Mortlock D.; Folger M.","The UKIRT Infrared Deep Sky Survey (UKIDSS) is a large-scale near-IR survey which aim is to cover 7500 square degrees of the Northern sky. The survey is carried out using the Wide Field Camera (WFCAM), with a field of view of 0.21 square degrees, mounted on the 3.8m United Kingdom Infra-red Telescope (UKIRT) in Hawaii. The project comprises five surveys (LAS, GCS, DXS, GPS and UDS). The Large Area Survey (LAS) covers an area of 4000 square degrees in high Galactic latitudes (extragalactic) in the four bands Y(1.0um) J(1.2um) H(1.6um) and K(2.2um) to a depth of K=18.4. This release 9 includes proper motions for a fraction (~1/6) of the sources. The Galactic Clusters Survey (GCS) aims to survey ten large open star clusters and star formation associations, covering a total of 1067 square degrees in the five bands Z (0.9um), Y(1.0um) J(1.2um) H(1.6um) and K(2.2um), plus a second pass in K for proper motions, to a depth of Z=20.4, Y=20.3, J=19.5, H=18.6, K=18.6. This release 9 includes proper motions for a fraction (~1/4) of the sources. The Deep Extragalactic Survey (DXS) aims to map 35 square degrees of sky to a 5-{sigma} point-source sensitivity of J=22.3 and K=20.8 in four carefully selected, multi-wavelength survey areas. The central regions of each field will also be mapped to H=21.8. The primary aim of the survey is to produce a photometric galaxy sample at a redshift of 1-2, within a volume comparable to that of the SDSS, selected in the same passband (rest frame optical). Details of the surveys can be found in the in the paper by Lawrence et al. (2007MNRAS.379.1599L), and at the UKIDSS Surveys site (http://www.ukidss.org/surveys/surveys.html). The data described here represent a subset of the UKIDSS data, limited to the public data and most representative columns. In the ""Byte-by-byte Description"" below the original names of the columns are given as bracketed names. Usage of the UKIDSS data: All users of UKIDSS data should include an acknowledgement in their publications as follows: ""This work is based in part on data obtained as part of the UKIRT Infrared Deep Sky Survey"".",2007MNRAS.379.1599L,https://cdsarc.cds.unistra.fr/viz-bin/cat/II/319,2013-04-16T12:29:07,2025-06-13T15:25:00,infrared,--


In [38]:
from astroquery.ukidss import Ukidss
import astropy.coordinates as coord
import astropy.units as u
table = Ukidss.query_region(coord.SkyCoord(12.947112,0.16112404,
                                           unit=(u.deg, u.deg),
                                           frame="icrs"),
                            programme_id="LAS", radius=1 * u.arcsec)
table


for i in range(len(DF)):
    ra, dec = DF['ra'][i], DF['dec'][i]
    table = Ukidss.query_region(coord.SkyCoord(ra,dec,
                                           unit=(u.deg, u.deg),
                                           frame="icrs"),
                            programme_id="LAS", radius=1 * u.arcsec)
    if len(table)!=0:
        print("Found: ", i)




Found:  0
Found:  1
Found:  2


Found:  5
Found:  6


Found:  10
Found:  11


Found:  17
Found:  18


Found:  20


...

Found:  87


Found:  90


Found:  93
Found:  94
Found:  95
Found:  96


Found:  98


Found:  104
Found:  105
Found:  106
